# Exercise 5: Star Schema for SRA Metadata
## BINFX410 — Chapter 10

---

## Learning Objectives
- Explain the difference between a normalized data lake schema and a dimensional star schema
- Identify fact and dimension tables in a genomics metadata context
- Build a star schema using Athena CTAS (CREATE TABLE AS SELECT) statements
- Write dbt-style SQL models demonstrating the ref() pattern
- Run analytical queries answering real bioinformatics metadata questions

**Estimated time:** 75 minutes  
**Dataset:** SRA Metadata (`s3://sra-pub-metadata-us-east-1`) — queried in-place via Athena  
**AWS services:** Athena, Glue Data Catalog, S3

## Background: The SRA Data Model and Why It Needs a Star Schema

The NCBI Sequence Read Archive (SRA) is the world's largest repository of raw sequencing data. Its data model reflects its submission-oriented design:

```
Study (SRP/PRJNA)           ← A research project
  └── Experiment (SRX)      ← A sequencing experiment (library + instrument)
        └── Run (SRR)       ← A sequencing run (actual FASTQ file)
              └── Sample (SRS) ← The biological sample
```

The SRA web interface is notoriously slow for bulk queries. The AWS Open Data Registry hosts SRA metadata as Parquet — we can query it directly with Athena at near-zero cost.

### Why Star Schema?

The normalized SRA model (4 tables, many JOINs) is hard to query for analytics:

> "How many RNA-seq runs were submitted from Homo sapiens in 2024, broken down by instrument platform?"

On the normalized model: 3 JOINs, complex filters.
On a star schema: `SELECT ... FROM fact_run JOIN dim_platform JOIN dim_sample WHERE ...`

### Our Star Schema Design

```
                    dim_study
                       │
dim_platform ──── fact_sequencing_run ──── dim_sample
                       │
                    dim_date
```

**fact_sequencing_run:** One row per SRR run. Measures: bases, spots, bytes.
**dim_sample:** organism, tissue, disease, sex, age
**dim_study:** BioProject accession, title, center, publication year
**dim_platform:** instrument model, library strategy, library source

## Section 1: Setup

**What this does:** This installs the required Python packages — `boto3`, `awswrangler`, `pandas`, and `matplotlib` — into the notebook environment. In a production SageMaker or Glue Studio notebook these would already be available in the managed kernel image.

```python
!pip install boto3 awswrangler pandas matplotlib --quiet
```

**What this does:** This cell configures all AWS clients and session objects for the exercise. On AWS, `boto3.Session(region_name='us-east-1')` authenticates via the IAM role attached to the compute instance, and constructs S3 paths for the student's data lake bucket and Athena query results location. The SRA Open Data database (`sra`) is referenced by name in the Glue catalog.

```python
import boto3
import awswrangler as wr
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

STUDENT_ID = "sab032226"  # CHANGE THIS
AWS_REGION = "us-east-1"
BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

session = boto3.Session(region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)

# SRA public metadata — query in-place, no copy needed
SRA_DATABASE = 'sra'  # Glue database name if using the SRA Open Data catalog

print("Setup complete.")
print(f"Note: SRA metadata is queried in-place from s3://sra-pub-metadata-us-east-1/")
```

## Section 2: Exploring SRA Metadata Structure

**What this does:** This cell documents the SRA metadata table schema (run, experiment, sample, study) for reference. In production, `wr.catalog.get_tables(database='sra')` would query the Glue Data Catalog to list the actual tables registered from `s3://sra-pub-metadata-us-east-1/`, returning their column definitions and S3 locations without incurring any data-transfer costs.

```python
# The SRA metadata is available as Athena-queryable Parquet
# First, let's look at what tables are available

# Option A: If the SRA Glue catalog is set up in your account
# tables = wr.catalog.get_tables(database='sra', boto3_session=session)

# Option B: We'll build our own schema from the SRA metadata format documentation
# and create representative data for the exercise

# The SRA metadata fields we care about:
sra_schema = {
    'run': ['acc', 'experiment', 'sample', 'study', 'releasedate', 'loaddate',
             'spots', 'bases', 'spots_with_mates', 'avgspotlen', 'size_MB',
             'assay_type', 'center_name', 'consent', 'runtype', 'platform'],
    'experiment': ['acc', 'study_accession', 'sample_accession', 'title',
                   'library_name', 'library_strategy', 'library_source',
                   'library_selection', 'library_layout', 'platform',
                   'instrument_model'],
    'sample': ['acc', 'organism', 'tax_id', 'biosample', 'tissue', 'sex',
                'disease', 'cell_type', 'cell_line', 'age'],
    'study': ['acc', 'bioproject', 'center_name', 'title', 'abstract',
               'received', 'type', 'submission']
}

print("SRA metadata schema (key tables and columns):")
for table, cols in sra_schema.items():
    print(f"\n  {table}:")
    print(f"    {', '.join(cols[:8])}...")
```

**What this does:** This builds a 200-row synthetic SRA run metadata DataFrame covering 8 instrument platforms, 5 organisms, 12 tissue types, and 9 disease categories with randomly generated accession numbers, base counts, and release dates. In production, this data would be read directly from the SRA Open Data Registry Parquet files at `s3://sra-pub-metadata-us-east-1/` via Athena — no copy required.

```python
# Build a representative SRA metadata sample for this exercise
# In production, this comes from s3://sra-pub-metadata-us-east-1/
import random
import numpy as np
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

platforms = [
    ('Illumina NovaSeq 6000', 'RNA-Seq', 'TRANSCRIPTOMIC'),
    ('Illumina NextSeq 500',  'RNA-Seq', 'TRANSCRIPTOMIC'),
    ('10x Genomics',         'RNA-Seq', 'TRANSCRIPTOMIC SINGLE CELL'),
    ('Illumina NovaSeq 6000', 'WGS',    'GENOMIC'),
    ('Illumina HiSeq 2500',  'WES',    'GENOMIC'),
    ('Illumina NovaSeq X',   'RNA-Seq', 'TRANSCRIPTOMIC'),
    ('PacBio Sequel II',     'WGS',    'GENOMIC'),
    ('Oxford Nanopore MinION','WGS',   'GENOMIC'),
]

organisms = [
    ('Homo sapiens', 9606),
    ('Mus musculus', 10090),
    ('Danio rerio', 7955),
    ('Arabidopsis thaliana', 3702),
    ('Saccharomyces cerevisiae', 4932),
]

tissues = ['Brain', 'Liver', 'Lung', 'Heart', 'Kidney', 'Blood', 'Skin',
           'Breast', 'Colon', 'Ovary', 'Prostate', 'None']

diseases = ['None', 'Breast cancer', 'Lung adenocarcinoma', 'Glioblastoma',
            'Type 2 diabetes', 'Alzheimers disease', 'COVID-19', 'None', 'None']

n_runs = 200
start_date = datetime(2019, 1, 1)

runs = []
for i in range(n_runs):
    platform_info = random.choice(platforms)
    organism_info = random.choice(organisms)
    release_date = start_date + timedelta(days=random.randint(0, 365*5))
    
    runs.append({
        'run_acc': f'SRR{10000000+i}',
        'experiment_acc': f'SRX{5000000+i}',
        'sample_acc': f'SRS{7000000+i//3}',
        'study_acc': f'SRP{1000+i//10}',
        'bioproject': f'PRJNA{500000+i//10}',
        'instrument_model': platform_info[0],
        'library_strategy': platform_info[1],
        'library_source': platform_info[2],
        'organism': organism_info[0],
        'tax_id': organism_info[1],
        'tissue': random.choice(tissues),
        'disease': random.choice(diseases),
        'sex': random.choice(['male', 'female', 'unknown', 'not determined']),
        'bases': random.randint(1_000_000, 200_000_000_000),
        'spots': random.randint(1_000_000, 2_000_000_000),
        'size_mb': random.randint(50, 15000),
        'release_year': release_date.year,
        'release_date': release_date.strftime('%Y-%m-%d'),
        'center_name': random.choice(['Broad Institute', 'EMBL-EBI', 'BGI', 'NIH', 'Sanger', 'NCI']),
    })

df_runs = pd.DataFrame(runs)
print(f"Generated {len(df_runs)} representative SRA runs")
print(f"Instrument breakdown:")
print(df_runs['instrument_model'].value_counts().head(8).to_string())
```

## Section 3: Loading SRA Data into Athena

**What this does:** This creates the `binfx410_sra` Glue database and writes the synthetic run metadata to S3 as Parquet at `bronze/sra_metadata/runs/`, registering it as `binfx410_sra.raw_runs` in the catalog via `awswrangler`. In production this staging step would be skipped — Athena would query the SRA Open Data tables directly.

```python
# Create the binfx410_sra database
try:
    glue.create_database(DatabaseInput={'Name': 'binfx410_sra',
                                         'Description': 'SRA metadata star schema'})
    print("Created database: binfx410_sra")
except glue.exceptions.AlreadyExistsException:
    print("Database binfx410_sra already exists.")

# Write the raw SRA run metadata to S3 as Parquet (staging table)
wr.s3.to_parquet(
    df=df_runs,
    path=f"s3://{BUCKET}/bronze/sra_metadata/runs/",
    dataset=True,
    database='binfx410_sra',
    table='raw_runs',
    boto3_session=session,
    mode='overwrite'
)
print(f"\nStaging table created: binfx410_sra.raw_runs ({len(df_runs)} rows)")
```

## Section 4: Building Dimension Tables via CTAS

**What this does:** This defines a `run_ctas()` helper function that wraps the Athena CTAS (CREATE TABLE AS SELECT) pattern: it drops any existing table, deletes the S3 data, submits a `CREATE TABLE ... WITH (format='PARQUET') AS SELECT ...` DDL statement via `wr.athena.start_query_execution()`, and verifies the row count. Each CTAS call on AWS creates a new Glue table and writes Parquet files to the specified S3 path in a single atomic Athena operation.

```python
# Helper: run a CTAS query and wait for it
def run_ctas(table_name, sql, database='binfx410_sra'):
    # Drop existing table if it exists
    try:
        wr.athena.start_query_execution(
            sql=f"DROP TABLE IF EXISTS {database}.{table_name}",
            database=database,
            s3_output=ATHENA_RESULTS,
            boto3_session=session,
            wait=True
        )
    except: pass

    # Clean up existing S3 data
    try:
        wr.s3.delete_objects(
            path=f"s3://{BUCKET}/gold/star_schema/{table_name}/",
            boto3_session=session
        )
    except: pass

    full_sql = f"""
    CREATE TABLE {database}.{table_name}
    WITH (
        format = 'PARQUET',
        parquet_compression = 'SNAPPY',
        external_location = 's3://{BUCKET}/gold/star_schema/{table_name}/'
    ) AS
    {sql}
    """

    # Use start_query_execution (not read_sql_query) for DDL statements.
    # read_sql_query wraps queries in its own CTAS by default, which produces
    # an invalid nested CTAS when the SQL is already a CREATE TABLE AS SELECT.
    wr.athena.start_query_execution(
        sql=full_sql,
        database=database,
        s3_output=ATHENA_RESULTS,
        boto3_session=session,
        wait=True
    )

    # Verify row count
    count = wr.athena.read_sql_query(
        sql=f"SELECT COUNT(*) as n FROM {database}.{table_name}",
        database=database,
        s3_output=ATHENA_RESULTS,
        boto3_session=session
    )
    print(f"  Created {table_name}: {count['n'].iloc[0]} rows")

print("CTAS helper defined.")
```

**What this does:** These three CTAS queries execute on Athena to build the star schema dimension tables. `dim_platform` creates 8 rows of unique instrument/strategy/source combinations with an `assay_category` label. `dim_sample` deduplicates sample accessions and adds an `organism_group` column. `dim_study` captures unique BioProject/study combinations. Each CTAS writes Snappy-compressed Parquet to `s3://{BUCKET}/gold/star_schema/{table}/` and registers the table in the Glue catalog.

```python
# DIM_PLATFORM: unique combinations of instrument, strategy, source
print("Building dim_platform...")
run_ctas('dim_platform', """
SELECT
    ROW_NUMBER() OVER (ORDER BY instrument_model, library_strategy) AS platform_key,
    instrument_model,
    library_strategy,
    library_source,
    CASE
        WHEN instrument_model LIKE '10x Genomics%' THEN 'Single-cell'
        WHEN library_strategy IN ('WGS', 'WES') THEN 'DNA-seq'
        WHEN library_strategy = 'RNA-Seq' THEN 'Bulk RNA-seq'
        ELSE 'Other'
    END AS assay_category
FROM (
    SELECT DISTINCT instrument_model, library_strategy, library_source
    FROM binfx410_sra.raw_runs
) t
""")

# DIM_SAMPLE: unique biological samples
print("Building dim_sample...")
run_ctas('dim_sample', """
SELECT
    sample_acc AS sample_key,
    organism,
    tax_id,
    tissue,
    disease,
    sex,
    CASE
        WHEN organism = 'Homo sapiens' THEN 'Human'
        WHEN organism = 'Mus musculus' THEN 'Mouse'
        ELSE 'Other'
    END AS organism_group
FROM (
    SELECT DISTINCT sample_acc, organism, tax_id, tissue, disease, sex
    FROM binfx410_sra.raw_runs
) t
""")

# DIM_STUDY: unique BioProjects
print("Building dim_study...")
run_ctas('dim_study', """
SELECT
    study_acc AS study_key,
    bioproject,
    center_name,
    release_year
FROM (
    SELECT DISTINCT study_acc, bioproject, center_name, release_year
    FROM binfx410_sra.raw_runs
) t
""")
```

**What this does:** This CTAS query builds the central `fact_sequencing_run` fact table on Athena — one row per SRR accession — including foreign keys to `dim_sample` and `dim_study`, a concatenated natural key to `dim_platform`, and quantitative measures (bases, spots, size_mb, avg_read_length). The resulting Parquet files are written to `s3://{BUCKET}/gold/star_schema/fact_sequencing_run/` and registered in the Glue catalog.

```python
# FACT_SEQUENCING_RUN: one row per SRR run with foreign keys and measures
print("Building fact_sequencing_run...")
run_ctas('fact_sequencing_run', f"""
SELECT
    r.run_acc AS run_key,
    r.sample_acc AS sample_fk,
    r.study_acc AS study_fk,
    CONCAT(r.instrument_model, '|', r.library_strategy, '|', r.library_source) AS platform_natural_key,
    r.release_year,
    r.release_date,
    -- Measures
    r.bases,
    r.spots,
    r.size_mb,
    ROUND(CAST(r.bases AS double) / NULLIF(r.spots, 0), 1) AS avg_read_length,
    r.instrument_model,
    r.library_strategy
FROM binfx410_sra.raw_runs r
""")

print("\nStar schema complete!")
print("Tables: dim_platform, dim_sample, dim_study, fact_sequencing_run")
```

## Section 5: Introduction to dbt SQL Models

dbt (data build tool) is the standard way to manage these SQL transformations in production. Instead of manually running CTAS queries, you define SQL files and dbt handles dependencies, testing, and documentation.

**What this does:** This writes two dbt SQL model files (`dim_sample.sql` and `fact_sequencing_run.sql`) to the local `dbt/models/` directory. In a real dbt project on AWS, running `dbt run` would compile these Jinja-templated SQL files, resolve the `{{ ref() }}` dependencies into a directed acyclic graph (DAG), and execute them in dependency order as Athena CTAS queries — eliminating the need to manage execution order manually.

```python
# Write dbt model SQL files to our repository
# These would be in the dbt/models/ directory

import os

os.makedirs('../dbt/models', exist_ok=True)

# dim_sample.sql — dbt model
dim_sample_sql = """
-- models/dim_sample.sql
-- dbt model: dimension table for biological samples
-- ref() creates a dependency on the raw_runs source table

{{ config(materialized='table', file_format='parquet') }}

SELECT
    sample_acc AS sample_key,
    organism,
    tax_id,
    tissue,
    disease,
    sex,
    CASE
        WHEN organism = 'Homo sapiens' THEN 'Human'
        WHEN organism = 'Mus musculus' THEN 'Mouse'
        ELSE 'Other'
    END AS organism_group
FROM (
    SELECT DISTINCT sample_acc, organism, tax_id, tissue, disease, sex
    FROM {{ ref('raw_runs') }}  -- dbt ref() creates tracked dependency
) t
"""

# fact_sequencing_run.sql — dbt model
fact_run_sql = """
-- models/fact_sequencing_run.sql
-- dbt model: fact table for sequencing runs
-- depends on dim_sample and dim_study via ref()

{{ config(materialized='table', file_format='parquet') }}

SELECT
    r.run_acc AS run_key,
    s.sample_key AS sample_fk,
    st.study_key AS study_fk,
    r.release_year,
    r.bases,
    r.spots,
    r.size_mb,
    r.instrument_model,
    r.library_strategy
FROM {{ ref('raw_runs') }} r
JOIN {{ ref('dim_sample') }} s ON r.sample_acc = s.sample_key
JOIN {{ ref('dim_study') }} st ON r.study_acc = st.study_key
"""

# Save dbt model files
for fname, content in [('dim_sample.sql', dim_sample_sql),
                        ('fact_sequencing_run.sql', fact_run_sql)]:
    with open(f'../dbt/models/{fname}', 'w') as f:
        f.write(content)
    print(f"Written: ../dbt/models/{fname}")

print()
print("Key dbt concept: {{ ref('raw_runs') }} creates a tracked dependency.")
print("dbt knows: fact_sequencing_run depends on dim_sample depends on raw_runs.")
print("Running 'dbt run' executes them in the correct order automatically.")
```

## Section 6: Analytical Queries on the Star Schema

**What this does:** This Athena query joins `fact_sequencing_run` to `dim_sample` and aggregates run counts and total bases by instrument model, filtering to RNA-seq runs from Homo sapiens. On AWS, Athena scans only the relevant Parquet partitions and returns results in seconds, billing per terabyte of data scanned.

```python
# Query 1: Which instrument platform has the most RNA-seq runs for Homo sapiens?
q1 = """
SELECT
    f.instrument_model,
    COUNT(*) AS run_count,
    SUM(f.bases) / 1e12 AS total_bases_tb
FROM binfx410_sra.fact_sequencing_run f
JOIN binfx410_sra.dim_sample s ON f.sample_fk = s.sample_key
WHERE f.library_strategy = 'RNA-Seq'
  AND s.organism = 'Homo sapiens'
GROUP BY f.instrument_model
ORDER BY run_count DESC
"""

df_q1 = wr.athena.read_sql_query(sql=q1, database='binfx410_sra',
                                  s3_output=ATHENA_RESULTS, boto3_session=session)
print("Q1: RNA-seq run count by instrument (Homo sapiens):")
df_q1
```

**What this does:** This Athena query computes year-over-year run counts and total data volume (GB) grouped by `release_year` and `library_strategy` for 2019–2024. On AWS, Athena executes this aggregation as a distributed scan of the `fact_sequencing_run` Parquet files, making it trivial to answer time-series questions that would require full table scans on a traditional relational database.

```python
# Query 2: Year-over-year run growth by library strategy
q2 = """
SELECT
    release_year,
    library_strategy,
    COUNT(*) AS run_count,
    ROUND(SUM(size_mb) / 1024.0, 1) AS total_gb
FROM binfx410_sra.fact_sequencing_run
WHERE release_year BETWEEN 2019 AND 2024
GROUP BY release_year, library_strategy
ORDER BY release_year, run_count DESC
"""

df_q2 = wr.athena.read_sql_query(sql=q2, database='binfx410_sra',
                                  s3_output=ATHENA_RESULTS, boto3_session=session)
print("Q2: Run count by year and library strategy:")
df_q2
```

**What this does:** This pivots the year-over-year query result into a wide DataFrame and renders a grouped bar chart of run counts by year and library strategy using matplotlib. In a SageMaker notebook, this plot would render inline; in a production pipeline it would be saved to S3 and embedded in a CloudWatch dashboard or Quarto report.

```python
# Visualize: run growth over time by strategy
pivot = df_q2.pivot_table(index='release_year', columns='library_strategy',
                           values='run_count', fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('SRA Run Count by Year and Library Strategy\n(Representative sample from star schema)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Runs')
ax.legend(title='Library Strategy', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('sra_run_growth.png', dpi=150, bbox_inches='tight')
plt.show()
```

**What this does:** This Athena query joins `fact_sequencing_run` to `dim_sample` and aggregates run counts and average run size by disease and library strategy, filtered to human disease samples. On AWS, this cross-table JOIN executes as a single distributed query against the Parquet star schema tables in S3, demonstrating the analytical power of the dimensional model.

```python
# Query 3: Disease-specific sequencing — which diseases have the most WGS data?
q3 = """
SELECT
    s.disease,
    f.library_strategy,
    COUNT(*) AS run_count,
    ROUND(AVG(f.size_mb) / 1024.0, 2) AS avg_gb_per_run
FROM binfx410_sra.fact_sequencing_run f
JOIN binfx410_sra.dim_sample s ON f.sample_fk = s.sample_key
WHERE s.disease != 'None'
  AND s.organism = 'Homo sapiens'
GROUP BY s.disease, f.library_strategy
ORDER BY run_count DESC
LIMIT 15
"""

df_q3 = wr.athena.read_sql_query(sql=q3, database='binfx410_sra',
                                  s3_output=ATHENA_RESULTS, boto3_session=session)
print("Q3: Top disease-specific sequencing projects:")
df_q3
```

## Exercise: YOUR CODE HERE

**What this does:** This is a student exercise cell. On AWS, completing it would submit a CTAS query to Athena that derives year, quarter, month, and an `is_recent` boolean from the `release_date` column in `fact_sequencing_run`, writing a new `dim_date` dimension table as Parquet to `s3://{BUCKET}/gold/star_schema/dim_date/`.

```python
# TASK 1: Create a dim_date dimension table
# Use CTAS to create a table with: year, quarter, month, is_recent (last 2 years)
# derived from the release_date in the fact table

# YOUR CODE HERE
run_ctas('dim_date', """
-- YOUR SQL HERE
""")
```

**What this does:** This is a student exercise cell. Completing it would submit an Athena query joining `fact_sequencing_run`, `dim_sample`, and `dim_platform` to find which tissue type has accumulated the most total sequenced bases from single-cell RNA-seq runs for Homo sapiens.

```python
# TASK 2: Write a query that answers:
# "Which tissue type has the most single-cell RNA-seq data (by total bases) for Homo sapiens?"

# YOUR CODE HERE
sc_query = """
-- YOUR SQL HERE
"""
# result = wr.athena.read_sql_query(...)
```

**What this does:** This is a student exercise cell. Completing it would submit an Athena query that joins `fact_sequencing_run` to `dim_sample` and calculates, for each `organism_group`, the percentage breakdown of runs by library strategy (RNA-Seq, WGS, WES), using `COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY organism_group)` window arithmetic.

```python
# TASK 3: The SRA contains data for many organisms.
# Write a query to find: for each organism_group (Human, Mouse, Other),
# what fraction of runs are RNA-seq vs WGS vs WES?
# Express as a percentage. Use the star schema tables.

# YOUR CODE HERE
```

## Reflection Questions

*(Double-click to edit)*

1. **Normalization vs denormalization:** The SRA's normalized model (4 tables, many FKs) is good for data integrity. The star schema is better for analytics. What would you lose if you moved to a fully denormalized model (one flat table with all columns)?

2. **Slowly Changing Dimensions (SCD):** A sample's `disease` annotation can be updated after submission (e.g., initial diagnosis was 'lung cancer', updated to 'lung adenocarcinoma'). How would you handle this in your `dim_sample` table? What SCD type would you use and why?

3. **dbt's `ref()` function** creates a dependency graph between models. If `dim_sample` changes (e.g., you add a column), dbt automatically knows to rebuild `fact_sequencing_run` too. Compare this to manually managing the CTAS statements we ran in this notebook. What problems does dbt solve?

4. **The star schema put `instrument_model` in both `dim_platform` and `fact_sequencing_run`.** Is this a design flaw? When is it acceptable to denormalize a dimension into the fact table?

5. **The SRA metadata is publicly queryable at no cost.** However, the actual sequencing data (FASTQ/BAM) costs money to download (egress fees). How does this economic asymmetry shape how you'd design a genomics data platform for a lab with a $5,000/year AWS budget?

*(Write answers here)*

1. 

2. 

3. 

4. 

5. 